In [1]:
import pandas as pd
import numpy as np
import glob
import yaml

import xml.etree.ElementTree as ET

In [2]:
with open("config_dataset_ff.yaml", "r") as stream:
    config_dataset = yaml.safe_load(stream)

metadata_path = config_dataset['metadata_path']
metadata_path

'data/metadata_ff.csv'

In [3]:
metadata = pd.read_csv(metadata_path)
metadata

,image_path,rna_path,case_id,sample_slide_id,sample_rna_id,sample_type,data_type_info,id_pair
0,TCGA-60-2712-01A-01-TS1.c5829f1c-421a-4498-b8c...,80ca4c12-21e8-45d1-8820-537b99bce32d.rna_seq.a...,TCGA-60-2712,TCGA-60-2712-01A,TCGA-60-2712-01A,Primary,TS,0
1,TCGA-56-7221-01A-01-TS1.97c4d16d-5160-46ac-8f4...,545f9937-b128-4f44-8b12-ded0fb79bf3f.rna_seq.a...,TCGA-56-7221,TCGA-56-7221-01A,TCGA-56-7221-01A,Primary,TS,1
2,TCGA-21-A5DI-01A-03-TS3.FD5286B2-AC59-425F-B94...,9b86812f-b1ee-4b6d-9691-8587f2487c4a.rna_seq.a...,TCGA-21-A5DI,TCGA-21-A5DI-01A,TCGA-21-A5DI-01A,Primary,TS,2
3,TCGA-43-7657-11A-01-TS1.7de37250-e538-4040-881...,4c84a3bd-3b6f-4627-a4d9-acd3b9669608.rna_seq.a...,TCGA-43-7657,TCGA-43-7657-11A,TCGA-43-7657-11A,Not Applicable,TS,3
4,TCGA-94-7033-01A-01-TS1.d38f20aa-3a65-4af3-a58...,23f1ad0c-c9d5-408f-bba8-1bb71364007b.rna_seq.a...,TCGA-94-7033,TCGA-94-7033-01A,TCGA-94-7033-01A,Primary,TS,4
...,...,...,...,...,...,...,...,...
458,TCGA-43-6773-11A-01-TS1.6e328690-6ad9-4d61-a42...,187da62b-d4a8-4c9e-a97e-ae5cb3a2c658.rna_seq.a...,TCGA-43-6773,TCGA-43-6773-11A,TCGA-43-6773-11A,Not Applicable,TS,458
459,TCGA-43-6143-01A-01-TS1.6716067a-f179-46cf-8d3...,3953b63a-1a63-4e97-b683-ae905b55e03e.rna_seq.a...,TCGA-43-6143,TCGA-43-6143-01A,TCGA-43-6143-01A,Primary,TS,459
460,TCGA-43-A56V-01A-01-TSA.6B4B05A2-0CAC-4779-9C8...,50f7bf08-3f8b-492b-afdc-40b4f35060bf.rna_seq.a...,TCGA-43-A56V,TCGA-43-A56V-01A,TCGA-43-A56V-01A,Primary,TS,460
461,TCGA-56-5897-01A-01-TS1.360c2b58-7187-4e91-a2e...,8175545d-b1a7-472e-aa7f-23adaece2ccd.rna_seq.a...,TCGA-56-5897,TCGA-56-5897-01A,TCGA-56-5897-01A,Primary,TS,461


In [4]:
def load_last_vist_day(sample_id):
    try:
        path =  glob.glob(f'data/*/*/*.{sample_id}.xml')[0]
        tree = ET.parse(path)
        root = tree.getroot()
        
        # Define the variable you're searching for
        variable_name = "vital_status"
        
        # Search for the variable in the XML tree
        for elem in root.iter():
            if "vital_status" in elem.tag:
                status = elem.text
            if "days_to_death" in elem.tag:
                days_to_death = elem.text 
            if "days_to_last_followup" in elem.tag:
                days_to_last_followup = elem.text
    
        if status == "Alive":
            return (sample_id, status, days_to_last_followup)
        else:
            return (sample_id, status, days_to_death)
    except:
        return (sample_id, np.nan, np.nan)

In [5]:
survival_metadata = metadata.case_id.apply(lambda x: load_last_vist_day(x))
survival_metadata = pd.DataFrame([(r) for r in survival_metadata.values], columns=["case_id", "censored", "event_time"])
survival_metadata

,case_id,censored,event_time
0,TCGA-60-2712,Dead,274
1,TCGA-56-7221,NaN,NaN
2,TCGA-21-A5DI,Alive,979
3,TCGA-43-7657,NaN,NaN
4,TCGA-94-7033,NaN,NaN
...,...,...,...
458,TCGA-43-6773,Dead,116
459,TCGA-43-6143,Alive,699
460,TCGA-43-A56V,Alive,366
461,TCGA-56-5897,Alive,378


In [6]:
survival_metadata = survival_metadata.drop_duplicates('case_id')
survival_metadata = survival_metadata[~survival_metadata.event_time.isna()]
survival_metadata = survival_metadata[survival_metadata.event_time.astype(int) > 0]
survival_metadata

,case_id,censored,event_time
0,TCGA-60-2712,Dead,274
2,TCGA-21-A5DI,Alive,979
6,TCGA-60-2715,Dead,1075
7,TCGA-NK-A5D1,Alive,511
11,TCGA-60-2698,Dead,311
...,...,...,...
457,TCGA-NC-A5HT,Alive,804
458,TCGA-43-6773,Dead,116
460,TCGA-43-A56V,Alive,366
461,TCGA-56-5897,Alive,378


In [7]:
survival_metadata.sort_values("case_id")

,case_id,censored,event_time
105,TCGA-18-3408,Dead,2304
445,TCGA-18-3412,Dead,345
437,TCGA-18-3414,Dead,716
434,TCGA-18-3415,Dead,2803
118,TCGA-18-3417,Dead,1097
...,...,...,...
238,TCGA-O2-A52S,Dead,387
365,TCGA-O2-A52V,Dead,1335
411,TCGA-O2-A52W,Dead,261
368,TCGA-O2-A5IB,Dead,340


In [8]:
survival_metadata.censored.value_counts()

censored
Alive    211
Dead     169
Name: count, dtype: int64

In [9]:
survival_metadata.to_csv(f'{metadata_path.replace(".csv", "")}_survival.csv', index=False)